In [1]:
import pandas as pd

# Extract
excel_path = "Coffee Shop Sales (2).xlsx"  # adjust path if needed

# Read the Transactions sheet
df = pd.read_excel(excel_path, sheet_name="Transactions")

# View the first few rows
df.head()

,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


In [3]:
df['transaction_date'] = pd.to_datetime(df['transaction_date']).dt.date
df['transaction_time'] = pd.to_datetime(df['transaction_time'], format='%H:%M:%S', errors='coerce').dt.time
df['total_amount'] = df['transaction_qty'] * df['unit_price']

dim_store = df[['store_id', 'store_location']].drop_duplicates().reset_index(drop=True)
dim_product = df[['product_id', 'product_category', 'product_type', 'product_detail']].drop_duplicates().reset_index(drop=True)

dim_date = pd.DataFrame({
    'date_key': pd.to_datetime(df['transaction_date']),
})
dim_date['year'] = dim_date['date_key'].dt.year
dim_date['month'] = dim_date['date_key'].dt.month
dim_date['day'] = dim_date['date_key'].dt.day
dim_date['weekday_name'] = dim_date['date_key'].dt.day_name()
dim_date = dim_date.drop_duplicates().reset_index(drop=True)

dim_time = pd.DataFrame({
    'time_key': pd.to_datetime(df['transaction_time'].astype(str), format='%H:%M:%S', errors='coerce')
})
dim_time['hour'] = dim_time['time_key'].dt.hour
dim_time['minute'] = dim_time['time_key'].dt.minute
dim_time['timeslot'] = dim_time['hour'].apply(
    lambda h: 'morning' if 5 <= h < 12 else
              'afternoon' if 12 <= h < 17 else
              'evening' if 17 <= h < 22 else 'night'
)
dim_time = dim_time.drop_duplicates().reset_index(drop=True)

fact_transactions = df[['transaction_id', 'transaction_date', 'transaction_time',
                        'store_id', 'product_id', 'transaction_qty',
                        'unit_price', 'total_amount']].copy()

fact_transactions.head()

,transaction_id,transaction_date,transaction_time,store_id,product_id,transaction_qty,unit_price,total_amount
0,1,2023-01-01,07:06:11,5,32,2,3.0,6.0
1,2,2023-01-01,07:08:56,5,57,2,3.1,6.2
2,3,2023-01-01,07:14:04,5,59,2,4.5,9.0
3,4,2023-01-01,07:20:24,5,22,1,2.0,2.0
4,5,2023-01-01,07:22:41,5,57,2,3.1,6.2


In [19]:
from pathlib import Path

OUTPUT_DIR = Path("./coffee_shop_csv_exports")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tables = {
    "dim_store.csv":        dim_store,
    "dim_product.csv":      dim_product,
    "dim_date.csv":         dim_date,
    "dim_time.csv":         dim_time,
    "fact_transactions.csv": fact_transactions,
}

for filename, df in tables.items():
    filepath = OUTPUT_DIR / filename
    df.to_csv(filepath, index=False, encoding='utf-8')
    print(f"Exported: {filepath}")

print("All tables exported as CSV files!")
print(f"Location: {OUTPUT_DIR.resolve()}")

Exported: coffee_shop_csv_exports\dim_store.csv
Exported: coffee_shop_csv_exports\dim_product.csv
Exported: coffee_shop_csv_exports\dim_date.csv
Exported: coffee_shop_csv_exports\dim_time.csv
Exported: coffee_shop_csv_exports\fact_transactions.csv
All tables exported as CSV files!
Location: C:\Users\Dell\Documents\Project Pipeline\coffee_shop_csv_exports


In [6]:
!git init

Initialized empty Git repository in C:/Users/Dell/Documents/Project Pipeline/.git/


In [7]:
!git checkout -b main

Switched to a new branch 'main'


In [8]:
!git checkout -b feature/etl-pipeline

Switched to a new branch 'feature/etl-pipeline'


In [9]:
!git add .
!git commit -m "Add ETL pipeline notebook + SQLite DB"

[feature/etl-pipeline (root-commit) 1e72021] Add ETL pipeline notebook + SQLite DB
 4 files changed, 441 insertions(+)
 create mode 100644 .ipynb_checkpoints/etl_pipeline-checkpoint.ipynb
 create mode 100644 Coffee Shop Sales (2).xlsx
 create mode 100644 coffee_shop.db
 create mode 100644 etl_pipeline.ipynb


In [11]:
!git remote add origin https://github.com/PhalePallo/data-pipeline-builder.git

error: remote origin already exists.


In [12]:
!git push -u origin feature/etl-pipeline

branch 'feature/etl-pipeline' set up to track 'origin/feature/etl-pipeline'.


remote: 
remote: Create a pull request for 'feature/etl-pipeline' on GitHub by visiting:        
remote:      https://github.com/PhalePallo/data-pipeline-builder/pull/new/feature/etl-pipeline        
remote: 
To https://github.com/PhalePallo/data-pipeline-builder.git
 * [new branch]      feature/etl-pipeline -> feature/etl-pipeline


In [20]:
!git branch

* feature/etl-pipeline


In [ ]:
!git add .